In [5]:
import pandas as pd
from src.data import DATA_DIR_INTERIM

from topic_gen.models import TRECTopic, Topics
from topic_gen.evaluate import CosineSimilarity, BertScore, JaccardIndex, TopicEvaluator, RelativeLength
import os
from src.data import get_dataset, load_topics_from_path

In [6]:
# Load generated qrels from path
BASE_DIR = DATA_DIR_INTERIM / "topics-robust"
predictions, names, metadata = load_topics_from_path(BASE_DIR)

[topic_gen] [WARNING] (data.py:88) Metadata not found for result 2025-11-30_17:25:43, skipping...


In [7]:
reference = Topics.load_ird_topics("disks45/nocr/trec-robust-2004")

In [8]:
results = TopicEvaluator.experiment(
    predictions=predictions,
    reference=reference,
    measures=[CosineSimilarity(), BertScore(), JaccardIndex(),
              RelativeLength()],
    names=names)

KeyboardInterrupt: 

In [ ]:
df = pd.DataFrame(results)
df = df.pivot(index=["name"], columns=["measure", "field"], values="value").reset_index()
metadata.columns = pd.MultiIndex.from_product([metadata.columns, ['']])
df = df.merge(metadata, left_on=[("name", "")], right_on=[('date', '')])

In [ ]:
df[[('model',''), ('prompt',''), ('nqueries',''), ('ndocspos',''), ('ndocsneg',''),
    ("CosineSimilarity","title"), ("BertScore","title"), ("JaccardIndex","title"), ("RelativeLength","title"),
    ("CosineSimilarity","description"), ("BertScore","description"), ("JaccardIndex","description"), ("RelativeLength","description"),
    ("CosineSimilarity","narrative"), ("BertScore","narrative"), ("JaccardIndex","narrative"), ("RelativeLength","narrative")]]

,model,prompt,nqueries,ndocspos,ndocsneg,CosineSimilarity,BertScore,JaccardIndex,RelativeLength,CosineSimilarity,BertScore,JaccardIndex,RelativeLength,CosineSimilarity,BertScore,JaccardIndex,RelativeLength
,,,,,,title,title,title,title,description,description,description,description,narrative,narrative,narrative,narrative
0,qwen3-30B-no-think,topic-query-docs-pos,4,4,0,0.772533,0.700411,0.60033,1.358148,0.752296,0.678409,0.714610,1.718295,0.639087,0.530333,0.645452,3.456388
1,qwen3-30B-no-think,topic-query-contrastive,1,4,4,0.794637,0.728543,0.63399,1.323356,0.725672,0.670311,0.705787,1.690979,0.639417,0.535488,0.641702,3.826371


In [ ]:
# Table
# Context / similarity lineplot
# Nqueries, ndocs vs similarity Heatmap
# examples

In [ ]:
def load_topics_and_run_evaluation(
        reference_topics_ird,
        regerated_topics_class,
        regenerated_topics_path,
        measures):

    # Load topics
    reference_topics = Topics.load_ird_topics(reference_topics_ird)
    generated_topics = Topics[regerated_topics_class].read_jsonl(
        regenerated_topics_path / "topics.jsonl")

    ref_topics = aligne_topics(generated_topics, reference_topics)
    reference_topics.topics = ref_topics

    # Run evaluation
    results = TopicEvaluator.evaluate(
        predictions=generated_topics,
        references=reference_topics,
        measures=measures,
        aggregate=True
    )

    # Present results
    scores = pd.DataFrame(results)
    scores["topic_id"] = scores["topic_id"].astype(str)
    scores = scores.merge(pd.DataFrame(reference_topics.topics).add_prefix(
        "ref_"), left_on="topic_id", right_on="ref_query_id")

    df = pd.DataFrame([row.model_dump()
                      for row in generated_topics.topics]).add_prefix("gen_")
    df["gen_topic_id"] = df["gen_topic_id"].astype(str)

    scores = scores.merge(df, left_on="topic_id", right_on="gen_topic_id")
    return scores